In [ ]:
# Example 1: public API surface — top-level exports and jupyter server extension hooks.
from jupyter_ai_router import RouterExtension, _jupyter_server_extension_points, _jupyter_labextension_paths

ext_points = _jupyter_server_extension_points()
lab_paths = _jupyter_labextension_paths()

print(f"extension points: {ext_points}")
print(f"labextension paths: {lab_paths}")
print(f"RouterExtension.name = {RouterExtension.name}")

assert isinstance(ext_points, list) and len(ext_points) == 1
assert ext_points[0]["module"] == "jupyter_ai_router"
assert ext_points[0]["app"] is RouterExtension
assert isinstance(lab_paths, list) and lab_paths[0]["dest"] == "@jupyter-ai/router"
assert RouterExtension.name == "jupyter_ai_router"

In [ ]:
# Example 2: utility helpers — get_first_word + is_persona.
from jupyter_ai_router.utils import get_first_word, is_persona

assert get_first_word("hello world") == "hello"
assert get_first_word("   /help me") == "/help"
assert get_first_word("   ") is None
assert get_first_word("") is None
assert get_first_word("single") == "single"

assert is_persona("jupyter-ai-personas-claude") is True
assert is_persona("jupyter-ai-personas") is True
assert is_persona("regular_user") is False
assert is_persona("") is False

print("get_first_word + is_persona behave as documented")

In [ ]:
# Example 3: router.matches_pattern — regex matching for slash command dispatch.
from jupyter_ai_router.router import matches_pattern

# Exact match.
assert matches_pattern("help", "help") is True
assert matches_pattern("helper", "help") is False

# Wildcard pattern.
assert matches_pattern("ai-generate", "ai-.*") is True
assert matches_pattern("ai-review", "ai-.*") is True
assert matches_pattern("help", "ai-.*") is False

# Alternation.
assert matches_pattern("export-json", "export-(json|csv)") is True
assert matches_pattern("export-csv", "export-(json|csv)") is True
assert matches_pattern("export-xml", "export-(json|csv)") is False

# Invalid regex must return False without raising.
assert matches_pattern("anything", "[unclosed") is False

print("matches_pattern dispatches as documented")

In [ ]:
# Example 4: MessageRouter — construct + register observer callbacks (no real YChat needed).
from jupyter_ai_router.router import MessageRouter

router = MessageRouter()

assert not hasattr(router, "observe_chat_reset"), (
    "MessageRouter.observe_chat_reset is back -- it was removed at 0.1.1; if upstream restored "
    "it, register it below again instead of leaving this assertion to fail"
)
assert not hasattr(router, "chat_reset_observers"), "chat_reset_observers is back"

# Register one of each surviving observer kind.
router.observe_chat_init(lambda room_id, ychat: None)
router.observe_chat_stop(lambda room_id: None)
router.observe_slash_cmd_msg("room-A", "help", lambda r, c, m: None)
router.observe_slash_cmd_msg("room-A", "ai-.*", lambda r, c, m: None)
router.observe_chat_msg("room-A", lambda r, m: None)

assert len(router.chat_init_observers) == 1
assert len(router.chat_stop_observers) == 1
assert set(router.slash_cmd_observers["room-A"].keys()) == {"help", "ai-.*"}
assert len(router.chat_msg_observers["room-A"]) == 1
assert router.message_observers == {}, router.message_observers
assert router.active_chats == {}
print("MessageRouter observer registration works")

In [ ]:
# Example 5: MessageRouter.cleanup() clears all registered observers.
from jupyter_ai_router.router import MessageRouter

router = MessageRouter()
router.observe_chat_init(lambda room_id, ychat: None)
router.observe_chat_stop(lambda room_id: None)
router.observe_slash_cmd_msg("room-X", "help", lambda r, c, m: None)
router.observe_chat_msg("room-X", lambda r, m: None)

# Sanity: observers are populated before cleanup.
assert router.chat_init_observers
assert router.chat_stop_observers
assert router.slash_cmd_observers
assert router.chat_msg_observers

# disconnect_chat is a no-op for rooms that were never connect_chat()'d.
router.disconnect_chat("room-never-connected")

router.cleanup()

assert router.chat_init_observers == []
assert router.chat_stop_observers == []
assert router.slash_cmd_observers == {}
assert router.chat_msg_observers == {}
assert router.active_chats == {}
print("MessageRouter.cleanup() resets all state")